# Web Scraping with BeautifulSoup (and selector gadget)

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

https://www.schmittchevrolet.com/new-vehicles/?_p=2

In [43]:
import requests
import pandas as pd
from bs4 import BeautifulSoup


r = requests.get('https://www.schmittchevrolet.com/new-vehicles/?_p=2')
soup = BeautifulSoup(r.content, 'html.parser')
#soup

content = soup.select('.price-block:nth-child(1) .price , .price span , .title-top , .title-bottom')
#content
for i in content:
    print(i.text)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [24]:
import asyncio
import random
import pandas as pd
from playwright.async_api import async_playwright

BASE_URL = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt={pt}"

async def scrape_inventory(max_pages=50, headless=True):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        page = await browser.new_page(viewport={"width": 1400, "height": 900})
        page.set_default_timeout(30000)

        all_rows = []

        for pt in range(1, max_pages + 1):
            await page.goto(BASE_URL.format(pt=pt), wait_until="domcontentloaded", timeout=60000)

            # Wait for at least one vehicle make/model to render
            try:
                await page.wait_for_selector(".vehicle-title__make-model", timeout=20000)
            except:
                print("Stop at page", pt, "no vehicles rendered")
                break

            rows = await page.evaluate(
                """() => {
                    const clean = (t) => t ? t.replace(/\\s+/g, ' ').trim() : null;

                    // This is the key change: start from make/model nodes (one per vehicle)
                    const mms = Array.from(document.querySelectorAll('.vehicle-title__make-model'));
                    const out = [];

                    const cardSelectors = [
                        '.vehicle-card',
                        '.vehicle-card__body',
                        '.vehicle-card__container',
                        'li',
                        'article',
                        'div'
                    ];

                    for (const mm of mms) {
                        // Walk up to a stable container that actually represents the vehicle card
                        let card = null;
                        for (const sel of cardSelectors) {
                            const c = mm.closest(sel);
                            if (c) { card = c; break; }
                        }
                        if (!card) continue;

                        const pick = (q) => {
                            const el = card.querySelector(q);
                            return el ? clean(el.textContent) : null;
                        };

                        // Prices: sites often show "highlight" as discount, not final price
                        const priceDiscount = pick('.vehiclePricingHighlightAmount, .priceBlocItemPriceValue');
                        const priceMain = pick(
                            '.vehiclePricingFinalPriceAmount, .vehiclePricingSalePriceAmount, .vehiclePricingMainAmount, .vehiclePricingAmount'
                        );

                        // Link: grab the first meaningful anchor inside the card
                        let href = null;
                        const a = card.querySelector("a[href]");
                        if (a) href = a.getAttribute("href");

                        const row = {
                            condition: pick('.vehicle-title__condition'),
                            year: pick('.vehicle-title__year'),
                            make_model: clean(mm.textContent),
                            trim: pick('.vehicle-title__trim'),
                            price_main: priceMain,
                            price_discount: priceDiscount,
                            href: href
                        };

                        out.push(row);
                    }

                    // De-dupe rows by key
                    const seen = new Set();
                    const deduped = [];
                    for (const r of out) {
                        const k = [r.year, r.make_model, r.trim, r.price_main, r.price_discount, r.href].join('|');
                        if (seen.has(k)) continue;
                        seen.add(k);
                        deduped.push(r);
                    }

                    return deduped;
                }"""
            )

            if not rows:
                print("Stop at page", pt, "0 rows")
                break

            # Normalize hrefs
            for r in rows:
                if r.get("href") and r["href"].startswith("/"):
                    r["href"] = "https://www.jackschmitt.com" + r["href"]
                r["page"] = pt

            all_rows.extend(rows)
            print("Page", pt, "vehicles", len(rows))

            await asyncio.sleep(random.uniform(0.9, 1.7))

        await browser.close()

    return all_rows

rows = await scrape_inventory(max_pages=10, headless=True)
df = pd.DataFrame(rows)

print(df.shape)
df.head(15)


Page 1 vehicles 21
Page 2 vehicles 13
Stop at page 3 no vehicles rendered
(34, 8)


,condition,year,make_model,trim,price_main,price_discount,href,page
0,New,2025,Chevrolet Trailblazer,None,None,"$3,000",https://www.jackschmitt.com/new-OFallon-2025-C...,1
1,New,2025,Chevrolet Trailblazer,ACTIV,None,"$3,000",https://www.jackschmitt.com/new-OFallon-2025-C...,1
2,New,2025,Chevrolet Silverado 2500 HD,WT,None,"$7,500",https://www.jackschmitt.com/new-OFallon-2025-C...,1
3,New,2025,Chevrolet Traverse,LT,None,"$4,500",https://www.jackschmitt.com/new-OFallon-2025-C...,1
4,New,2025,Chevrolet Silverado 1500,RST,None,"$13,000",https://www.jackschmitt.com/new-OFallon-2025-C...,1
5,New,2025,Chevrolet Silverado 1500,None,None,"$11,500",https://www.jackschmitt.com/new-OFallon-2025-C...,1
6,New,2026,Chevrolet Trailblazer,None,None,"$1,016",https://www.jackschmitt.com/new-OFallon-2026-C...,1
7,New,2026,Chevrolet Silverado 1500,None,None,"$11,250",https://www.jackschmitt.com/new-OFallon-2026-C...,1
8,New,2026,Chevrolet Equinox,None,None,"$4,500",https://www.jackschmitt.com/new-OFallon-2026-C...,1
9,New,2026,Chevrolet Suburban,Premier,None,"$6,052",https://www.jackschmitt.com/new-OFallon-2026-C...,1


In [6]:
r = requests.get('https://www.bommaritochevysouth.com/searchnew.aspx?Year=2024&make=Chevrolet&ModelAndTrim=Equinox')
soup = BeautifulSoup(r.content, 'html.parser')
#soup

In [8]:
content = soup.select('.ca-gpc-close-button-primary , .featuredPrice .vehiclePricingHighlightAmount')
#content
for i in content:
    print(i.text)

$22,999
$22,999
$22,999
$22,999
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,039
$23,094
$23,094
$23,094
$23,094
$23,319
$23,319
$23,319
$23,319
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664
$23,664


In [12]:
content2 = soup.select(".vehiclePricingHighlightAmount")
#content2
for i in content2:
    print(i.text)

$6,751
$22,999
$6,751
$22,999
$6,751
$22,999
$6,751
$22,999
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,039
$6,751
$23,094
$6,751
$23,094
$6,751
$23,094
$6,751
$23,094
$6,751
$23,319
$6,751
$23,319
$6,751
$23,319
$6,751
$23,319
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664
$6,751
$23,664


In [8]:
r = requests.get('https://www.bommaritochevysouth.com/searchnew.aspx?Year=2024&make=Chevrolet&ModelAndTrim=Equinox')
soup = BeautifulSoup(r.content, 'html.parser')
#soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="px-captcha" name="description"/>
<title>Access to this page has been denied</title>
</head>
<body>
<script>
    /* PerimeterX assignments */
    window._pxVid = '';
    window._pxUuid = 'd19a9ce2-f8f9-11ee-beaa-6a1b379ceb35';
    window._pxAppId = 'PXHYx10rg3';
    window._pxMobile = false;
    window._pxHostUrl = '/HYx10rg3/xhr';
    window._pxCustomLogo = 'https://www.zillowstatic.com/s3/pfs/static/z-logo-default.svg';
    window._pxJsClientSrc = '/HYx10rg3/init.js';
    window._pxFirstPartyEnabled = true;
    var pxCaptchaSrc = '/HYx10rg3/captcha/captcha.js?a=c&u=d19a9ce2-f8f9-11ee-beaa-6a1b379ceb35&v=&m=0';

    var script = document.createElement('script');
    script.src = pxCaptchaSrc;
    script.onerror = function () {
        script = document.createElement('script');
        script.src = 'https://captcha.px-cloud.net/PXHYx10rg3/

In [34]:
#Scraping a local file since Zillow detected direct scraping - so just download the html and then scrape!

with open('zillow.html', 'r', encoding='utf8') as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, 'lxml')
content = soup.select('abbr , b , .djgngA')

data = []

for i in content:
    data.append(i.text)

data

def chunk_list(lst, n):
    for i in range(0, len(data), n):
        yield data[i:i + n]

# Split the list into chunks of 7 items each
chunked_list = list(chunk_list(data, 7))

# Create a DataFrame from the chunked list
df = pd.DataFrame(chunked_list)

print(df)

          0  1    2  3   4      5     6
0  $295,000  4  bds  3  ba  2,502  sqft
1  $215,000  3  bds  2  ba  1,144  sqft
2  $500,000  4  bds  4  ba  3,204  sqft
3  $265,000  3  bds  2  ba  1,815  sqft
4  $289,000  3  bds  2  ba  1,752  sqft
5  $248,900  3  bds  2  ba  1,092  sqft
6  $189,900  3  bds  2  ba    925  sqft
7  $275,000  3  bds  3  ba  1,214  sqft
8  $444,900  4  bds  3  ba  2,548  sqft


In [40]:
#Scraping a local file since Zillow detected direct scraping - so just download the html and then scrape!

with open('/Users/robertwrobel/Downloads/zillow2.html', 'r', encoding='utf8') as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, 'lxml')
content = soup.select('.eKNMXc li , abbr , .djgngA')

data = []

for i in content:
    data.append(i.text)

data

def chunk_list(lst, n):
    for i in range(0, len(data), n):
        yield data[i:i + n]

chunked_list = list(chunk_list(data, 7))

df = pd.DataFrame(chunked_list)

print(df)

          0      1    2     3   4           5     6
0  $295,000  4 bds  bds  3 ba  ba  2,502 sqft  sqft
1  $215,000  3 bds  bds  2 ba  ba  1,144 sqft  sqft
2  $500,000  4 bds  bds  4 ba  ba  3,204 sqft  sqft
3  $265,000  3 bds  bds  2 ba  ba  1,815 sqft  sqft
4  $289,000  3 bds  bds  2 ba  ba  1,752 sqft  sqft
5  $248,900  3 bds  bds  2 ba  ba  1,092 sqft  sqft
6  $189,900  3 bds  bds  2 ba  ba    925 sqft  sqft
7  $275,000  3 bds  bds  3 ba  ba  1,214 sqft  sqft
8  $444,900  4 bds  bds  3 ba  ba  2,548 sqft  sqft


In [32]:
def chunk_list(lst, n):
    for i in range(0, len(data), n):
        yield data[i:i + n]

#Split the list into chunks of 7 items each
chunked_list = list(chunk_list(data, 7))

#Create a DataFrame from the chunked list
df = pd.DataFrame(chunked_list)

print(df)

          0  1    2  3   4      5     6
0  $295,000  4  bds  3  ba  2,502  sqft
1  $215,000  3  bds  2  ba  1,144  sqft
2  $500,000  4  bds  4  ba  3,204  sqft
3  $265,000  3  bds  2  ba  1,815  sqft
4  $289,000  3  bds  2  ba  1,752  sqft
5  $248,900  3  bds  2  ba  1,092  sqft
6  $189,900  3  bds  2  ba    925  sqft
7  $275,000  3  bds  3  ba  1,214  sqft
8  $444,900  4  bds  3  ba  2,548  sqft


In [8]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

r = requests.get('https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=2')
soup = BeautifulSoup(r.content, 'html.parser')


content = soup.select('.priceBlocItemPriceValue , .vehiclePricingHighlightAmount , .vehicle-title__year , .vehicle-title__condition')
content
for i in content:
    print(i.text)

In [5]:
content = soup.select('.vertical-stack > .price-block .price , .title-bottom , .title-top')
content
for i in content:
    print(i.text)

In [11]:
import requests

url = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=2"
headers = {"User-Agent": "Mozilla/5.0", "Accept-Language": "en-US,en;q=0.9"}

r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()

print("status:", r.status_code)
print("final url:", r.url)
print("len:", len(r.text))

needles = [
    "vehicle-title__make-model",
    "vehicle-title__trim",
    "vehicle-title__year",
    "vehicle-title__condition",
    "priceBlocItemPriceValue",
    "vehiclePricingHighlightAmount",
]
for n in needles:
    print(n, "->", n in r.text)


status: 200
final url: https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=2
len: 662154
vehicle-title__make-model -> True
vehicle-title__trim -> True
vehicle-title__year -> True
vehicle-title__condition -> True
priceBlocItemPriceValue -> True
vehiclePricingHighlightAmount -> True


In [12]:
import requests
from bs4 import BeautifulSoup

url = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=2"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

html = requests.get(url, headers=headers, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

cards = soup.select("div.vehicle-card__body")
print("vehicle cards:", len(cards))


vehicle cards: 0


In [14]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

URL = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=2"

def txt(el):
    return el.get_text(" ", strip=True) if el else None

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URL, wait_until="networkidle", timeout=60000)

        html = await page.content()
        await browser.close()

    soup = BeautifulSoup(html, "html.parser")

    # You can swap this selector once you confirm actual rendered markup
    cards = soup.select("div.vehicle-card__body")
    print("cards:", len(cards))

    rows = []
    for c in cards:
        rows.append({
            "condition": txt(c.select_one(".vehicle-title__condition")),
            "year": txt(c.select_one(".vehicle-title__year")),
            "make_model": txt(c.select_one(".vehicle-title__make-model")),
            "trim": txt(c.select_one(".vehicle-title__trim")),
            "price": txt(c.select_one(".vehiclePricingHighlightAmount")),
        })

    print(rows[:3])
    return rows

rows = await run()


Error: BrowserType.launch: Executable doesn't exist at /Users/robertwrobel/Library/Caches/ms-playwright/chromium_headless_shell-1200/chrome-headless-shell-mac-arm64/chrome-headless-shell
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     playwright install                                     ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝

In [19]:
import re
from playwright.async_api import async_playwright

URL = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=1"

async def find_endpoints():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled",
            ],
        )
        page = await browser.new_page()

        # Make timeouts sane
        page.set_default_timeout(60000)

        # Load without "networkidle"
        await page.goto(URL, wait_until="domcontentloaded", timeout=60000)

        # Give the app time to fetch inventory and render
        await page.wait_for_timeout(8000)

        # Pull all resources the page has requested
        resources = await page.evaluate("""
            () => performance.getEntriesByType('resource').map(r => ({
                name: r.name,
                initiatorType: r.initiatorType
            }))
        """)

        # Keep only likely data calls
        xhr = [r["name"] for r in resources if r["initiatorType"] in ("fetch", "xmlhttprequest")]
        xhr = sorted(set(xhr))

        # Optional: save proof the page actually rendered
        await page.screenshot(path="jackschmitt_pt1.png", full_page=True)

        await browser.close()

    print("XHR/fetch urls:", len(xhr))
    for u in xhr:
        if re.search(r"(inventory|vehicle|search|srp|api|stock|vin|results)", u, re.I):
            print(u)
    print("\nSaved screenshot: jackschmitt_pt1.png")
    return xhr

xhr_urls = await find_endpoints()


XHR/fetch urls: 18
https://api.dealeron.com/api/harmoniq/data/28807?_=1768242600
https://www.google-analytics.com/g/collect?v=2&tid=G-497VVQ4D7Q&gtm=45je6180v9180523487z89180517606za20gzb9180517606zd9180517606&_p=1768242609574&gcd=13l3l3l3l1l1&npa=0&dma=0&cid=463627827.1768242610&ul=en-us&sr=1280x720&uaa=arm&uab=64&uafvl=HeadlessChrome%3B143.0.7499.4%7CChromium%3B143.0.7499.4%7CNot%2520A(Brand%3B24.0.0.0&uamb=0&uam=&uap=macOS&uapv=26.2.0&uaw=0&are=1&frm=0&pscdl=noapi&_s=1&tag_exp=103116026~103200004~104527906~104528501~104684208~104684211~105391253~115616985~115938465~115938468~116514482~116682876~116744867&sid=1768242610&sct=1&seg=0&dl=https%3A%2F%2Fwww.jackschmitt.com%2Fsearchnew.aspx%3Fpn%3D96%26pt%3D1&dt=Chevrolet%20New%20Vehicle%20Inventory%20Search%20in%20OFALLON%20%7C%20Chevrolet%20dealership%20in%20OFALLON%20IL&en=page_view&_fv=1&_nsi=1&_ss=1&tfd=1186
https://www.jackschmitt.com/api/formsaa/28807/dates?addDays=0&totalDays=90&cmsHours=Sales
https://www.jackschmitt.com/api/formsa

In [20]:
import base64
import time
import random
import requests
import pandas as pd

SITE_ID = 28807
SRP_ID  = 3011315
HOST    = "www.jackschmitt.com"

API = f"https://{HOST}/api/vhcliaa/vehicle-pages/cosmos/srp/vehicles/{SITE_ID}/{SRP_ID}"
FILTERS_API = f"https://{HOST}/api/vhcliaa/vehicle-pages/cosmos/srp/filters/{SITE_ID}/{SRP_ID}"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": f"https://{HOST}/searchnew.aspx?pn=96&pt=1",
})

def b64(s: str) -> str:
    return base64.b64encode(s.encode("utf-8")).decode("utf-8")

def polite_sleep():
    time.sleep(random.uniform(0.8, 1.6))

def fetch_filters(pn=96, pt=1, base_filter="type='n'"):
    params = {
        "pn": pn,
        "pt": pt,
        "host": HOST,
        "baseFilter": b64(base_filter),
    }
    r = session.get(FILTERS_API, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_page(pn=96, pt=1, base_filter="type='n'"):
    params = {
        "pn": pn,
        "pt": pt,
        "host": HOST,
        "baseFilter": b64(base_filter),
        "displayCardsShown": "NaN",
    }
    r = session.get(API, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def scrape_all(pn=96, base_filter="type='n'", max_pages=200):
    rows = []
    for pt in range(1, max_pages + 1):
        data = fetch_page(pn=pn, pt=pt, base_filter=base_filter)

        # We do not know the exact key name yet, so handle common shapes
        items = None
        if isinstance(data, dict):
            for k in ["vehicles", "results", "items", "data", "list"]:
                if k in data and isinstance(data[k], list):
                    items = data[k]
                    break

        # If the response is already a list, that is our items
        if items is None and isinstance(data, list):
            items = data

        if not items:
            print("Stop at page", pt)
            break

        print("Page", pt, "items", len(items))
        rows.extend(items)
        polite_sleep()

    return rows

# 1) quick test on page 1
test = fetch_page(pn=96, pt=1, base_filter="type='n'")
print("Top-level keys:", list(test.keys())[:30] if isinstance(test, dict) else type(test))

# 2) full scrape
new_rows = scrape_all(pn=96, base_filter="type='n'", max_pages=200)
df_new = pd.json_normalize(new_rows)
print(df_new.shape)
df_new.head(5)


Top-level keys: ['ConditionalBlockHtml', 'Paging', 'SrpScrollType', 'DisplayCards', 'TpiScripts', 'TdgSaveHeartVehicles', 'SuggestionRedirectModel']
Stop at page 1
(0, 0)


""


In [26]:
import base64
import time
import random
import requests
from bs4 import BeautifulSoup

HOST = "www.jackschmitt.com"
SITE_ID = 28807
SRP_ID = 3011315

API = f"https://{HOST}/api/vhcliaa/vehicle-pages/cosmos/srp/vehicles/{SITE_ID}/{SRP_ID}"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": f"https://{HOST}/searchnew.aspx?pn=96&pt=1",
})

def b64(s):
    return base64.b64encode(s.encode()).decode()

def sleep():
    time.sleep(random.uniform(0.8, 1.4))


In [28]:
def parse_cards(html):
    soup = BeautifulSoup(html, "html.parser")
    cards = []

    for mm in soup.select(".vehicle-title__make-model"):
        card = mm.find_parent("div", class_=lambda c: c and "vehicle-card" in c)
        if not card:
            continue

        def txt(sel):
            el = card.select_one(sel)
            return el.get_text(" ", strip=True) if el else None

        price = txt(".vehiclePricingFinalPriceAmount") \
                or txt(".vehiclePricingSalePriceAmount") \
                or txt(".vehiclePricingHighlightAmount")

        cards.append({
            "condition": txt(".vehicle-title__condition"),
            "year": txt(".vehicle-title__year"),
            "make_model": txt(".vehicle-title__make-model"),
            "trim": txt(".vehicle-title__trim"),
            "price_displayed": price,
            "href": (
                "https://www.jackschmitt.com" + a["href"]
                if (a := card.select_one("a[href^='/new']"))
                else None
            ),
        })

    return cards


In [29]:
def fetch_all_new(pn=96):
    all_cards = []
    shown = 0
    pt = 1  # stays at 1 or 2 only

    while True:
        params = {
            "pn": pn,
            "pt": pt,
            "host": HOST,
            "baseFilter": b64("type='n'"),
            "displayCardsShown": shown if shown else "NaN",
        }

        r = session.get(API, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        html = data.get("ConditionalBlockHtml", "")
        new_cards = parse_cards(html)

        print(f"Fetched {len(new_cards)} cards (shown={shown})")

        if not new_cards:
            break

        all_cards.extend(new_cards)
        shown += len(new_cards)

        # after first call, pt must be 2
        pt = 2
        sleep()

    return all_cards


In [33]:
import asyncio
import random
import pandas as pd
from playwright.async_api import async_playwright

URL = "https://www.jackschmitt.com/searchnew.aspx?pn=96&pt=1"

async def scrape_srp_exact(max_scroll_loops=220, headless=True):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        page = await browser.new_page(viewport={"width": 1500, "height": 950})
        page.set_default_timeout(30000)

        await page.goto(URL, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_selector(".vehicle-card__body", timeout=25000)

        # If the page tells us total vehicles, use it as a target
        target_total = await page.evaluate("""
        () => {
            const el = Array.from(document.querySelectorAll("body *"))
              .find(x => x.textContent && x.textContent.match(/Showing\\s+All\\s+\\d+\\s+Vehicles/i));
            if (!el) return null;
            const m = el.textContent.match(/Showing\\s+All\\s+(\\d+)\\s+Vehicles/i);
            return m ? parseInt(m[1], 10) : null;
        }
        """)
        if target_total:
            print("Target total from page:", target_total)

        # Scroll until count stabilizes or we reach target
        prev = 0
        stable = 0
        for i in range(max_scroll_loops):
            count = await page.eval_on_selector_all(".vehicle-card__body", "els => els.length")

            if target_total and count >= target_total:
                break

            if count == prev:
                stable += 1
            else:
                stable = 0
            if stable >= 10:
                break

            prev = count
            await page.evaluate("window.scrollBy(0, Math.floor(window.innerHeight * 0.9));")
            await page.wait_for_timeout(900)

        final_count = await page.eval_on_selector_all(".vehicle-card__body", "els => els.length")
        print("Vehicle cards rendered:", final_count)

        # Expand pricing on all cards so MSRP line shows if it is behind "More"
        await page.evaluate("""
        () => {
            document.querySelectorAll('.vehiclePricingMoreLessButton').forEach(btn => {
                const label = (btn.textContent || '').toLowerCase();
                if (label.includes('more')) btn.click();
            });
        }
        """)
        await page.wait_for_timeout(1200)

        rows = await page.evaluate("""
        () => {
            const clean = t => t ? t.replace(/\\s+/g,' ').trim() : null;

            const cards = Array.from(document.querySelectorAll('.vehicle-card__body'));
            const out = [];

            for (const card of cards) {
                const pick = (sel) => {
                    const el = card.querySelector(sel);
                    return el ? clean(el.textContent) : null;
                };

                // VDP link
                const a = card.querySelector("a.hero-carousel__item--viewvehicle[href]") ||
                          card.querySelector("a[href*='/new-'][href]") ||
                          card.querySelector("a[href^='/new'][href]");
                let href = a ? a.getAttribute("href") : null;
                if (href && href.startsWith("/")) href = "https://www.jackschmitt.com" + href;

                // Discount highlight (dealer discount, rebates, etc)
                const discount = pick(".vehiclePricingHighlight.dealerDiscount .vehiclePricingHighlightAmount") ||
                                 pick(".vehiclePricingHighlightAmount.text-success");

                // Price rows map (label -> value), using the exact classes you surfaced
                const priceMap = {};
                card.querySelectorAll("li.priceBlockItemPrice").forEach(li => {
                    const lab = li.querySelector(".priceBlocItemPriceLabel");
                    const val = li.querySelector(".priceBlocItemPriceValue");
                    const k = lab ? clean(lab.textContent).replace(/:$/, "") : null;
                    const v = val ? clean(val.textContent) : null;
                    if (k && v) priceMap[k.toLowerCase()] = v;
                });

                // These label names vary, so we check several common ones
                const sale_price =
                    priceMap["sale price"] ||
                    priceMap["internet price"] ||
                    priceMap["final price"] ||
                    priceMap["your price"] ||
                    priceMap["price"];

                const msrp =
                    priceMap["msrp"] ||
                    priceMap["sticker price"] ||
                    priceMap["retail price"];

                out.push({
                    condition: pick(".vehicle-title__condition"),
                    year: pick(".vehicle-title__year"),
                    make_model: pick(".vehicle-title__make-model"),
                    trim: pick(".vehicle-title__trim"),
                    msrp: msrp,
                    sale_price: sale_price,
                    discount: discount,
                    href: href
                });
            }

            // filter obvious empties
            return out.filter(r => r.make_model || r.href);
        }
        """)

        await browser.close()
        return rows

rows = await scrape_srp_exact(headless=True)
df = pd.DataFrame(rows)
print(df.shape)
df.head(20)


Target total from page: 109
Vehicle cards rendered: 96
(96, 8)


,condition,year,make_model,trim,msrp,sale_price,discount,href
0,New,2025,Chevrolet Trailblazer,None,"$34,220","$31,220","$3,000",https://www.jackschmitt.com/new-OFallon-2025-C...
1,New,2025,Chevrolet Trailblazer,ACTIV,"$34,075","$31,075","$3,000",https://www.jackschmitt.com/new-OFallon-2025-C...
2,New,2025,Chevrolet Silverado 2500 HD,WT,"$50,435","$42,935","$7,500",https://www.jackschmitt.com/new-OFallon-2025-C...
3,New,2025,Chevrolet Traverse,LT,"$44,180","$39,680","$4,500",https://www.jackschmitt.com/new-OFallon-2025-C...
4,New,2025,Chevrolet Silverado 1500,RST,"$64,125","$51,125","$13,000",https://www.jackschmitt.com/new-OFallon-2025-C...
5,New,2025,Chevrolet Silverado 1500,None,"$47,260","$35,760","$11,500",https://www.jackschmitt.com/new-OFallon-2025-C...
6,New,2026,Chevrolet Trailblazer,None,"$26,415","$25,399","$1,016",https://www.jackschmitt.com/new-OFallon-2026-C...
7,New,2026,Chevrolet Silverado 1500,None,"$62,505","$51,255","$11,250",https://www.jackschmitt.com/new-OFallon-2026-C...
8,New,2026,Chevrolet Equinox,None,"$37,665","$33,165","$4,500",https://www.jackschmitt.com/new-OFallon-2026-C...
9,New,2026,Chevrolet Suburban,Premier,"$88,400","$82,348","$6,052",https://www.jackschmitt.com/new-OFallon-2026-C...


In [41]:
import numpy as np
import re

# -------------------------
# 1) Helpers
# -------------------------
MONEY_RE = re.compile(r"\$[\s]*([0-9]{1,3}(?:,[0-9]{3})+|[0-9]+)")

def money_to_int(x):
    if x is None:
        return np.nan
    s = str(x).strip()
    m = MONEY_RE.search(s)
    if not m:
        return np.nan
    return float(int(m.group(1).replace(",", "")))

# -------------------------
# 2) Clean + numeric columns
# -------------------------
df = df.copy()

df["model"] = (
    df["make_model"]
      .astype(str)
      .str.replace(r"^Chevrolet\s+", "", regex=True)
      .str.strip()
)

df["msrp_num"] = df["msrp"].apply(money_to_int)
df["sale_price_num"] = df["sale_price"].apply(money_to_int)
df["discount_num"] = df["discount"].apply(money_to_int)

# Compute discount when missing but MSRP and sale price exist
mask = df["discount_num"].isna() & df["msrp_num"].notna() & df["sale_price_num"].notna()
df.loc[mask, "discount_num"] = df.loc[mask, "msrp_num"] - df.loc[mask, "sale_price_num"]

# Discount percent of MSRP
df["discount_pct"] = df["discount_num"] / df["msrp_num"]

# Normalize trim
df["trim_norm"] = df["trim"].fillna("Unknown").astype(str).str.strip().replace({"None": "Unknown", "nan": "Unknown"})

# -------------------------
# 3) Tables you asked for
# -------------------------

# Total vehicles
total_vehicles_df = pd.DataFrame({"total_vehicles": [len(df)]})

# Count by model + pct of inventory
count_by_model = (
    df.groupby("model", as_index=False)
      .agg(count=("model", "count"))
      .sort_values("count", ascending=False)
)
total = count_by_model["count"].sum()
count_by_model["pct_of_inventory"] = count_by_model["count"] / total

# Count by model + trim (DataFrame)
count_by_model_trim = (
    df.groupby(["model", "trim_norm"], as_index=False)
      .agg(count=("trim_norm", "count"))
      .sort_values(["model", "count"], ascending=[True, False])
)
count_by_model_trim["pct_within_model"] = (
    count_by_model_trim["count"] /
    count_by_model_trim.groupby("model")["count"].transform("sum")
)

# Avg discount by model
discount_by_model = (
    df.groupby("model", as_index=False)
      .agg(
          vehicles=("model", "count"),
          avg_discount=("discount_num", "mean"),
          median_discount=("discount_num", "median"),
          avg_msrp=("msrp_num", "mean"),
          avg_sale_price=("sale_price_num", "mean"),
          avg_discount_pct=("discount_pct", "mean"),
          median_discount_pct=("discount_pct", "median"),
      )
      .sort_values("avg_discount", ascending=False)
)

# Avg discount by model + trim
discount_by_model_trim = (
    df.groupby(["model", "trim_norm"], as_index=False)
      .agg(
          vehicles=("trim_norm", "count"),
          avg_discount=("discount_num", "mean"),
          median_discount=("discount_num", "median"),
          avg_msrp=("msrp_num", "mean"),
          avg_sale_price=("sale_price_num", "mean"),
          avg_discount_pct=("discount_pct", "mean"),
          median_discount_pct=("discount_pct", "median"),
      )
      .sort_values(["model", "avg_discount"], ascending=[True, False])
)

# Optional: quick sanity table of weird rows
weird_rows = df[
    (df["discount_num"] < 0) |
    (df["msrp_num"].notna() & df["discount_num"].notna() & (df["discount_num"] > df["msrp_num"] * 0.5))
][["model", "trim_norm", "msrp", "sale_price", "discount", "href"]].copy()

# -------------------------
# 4) Write Excel workbook
# -------------------------
out_path = "jackschmitt_srp_tables.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    total_vehicles_df.to_excel(writer, sheet_name="total_vehicles", index=False)
    df.to_excel(writer, sheet_name="raw_inventory", index=False)
    count_by_model.to_excel(writer, sheet_name="count_by_model", index=False)
    count_by_model_trim.to_excel(writer, sheet_name="count_by_model_trim", index=False)
    discount_by_model.to_excel(writer, sheet_name="discount_by_model", index=False)
    discount_by_model_trim.to_excel(writer, sheet_name="discount_by_model_trim", index=False)
    weird_rows.to_excel(writer, sheet_name="sanity_weird_rows", index=False)

print("Wrote:", out_path)
print("Total vehicles:", len(df))


Wrote: jackschmitt_srp_tables.xlsx
Total vehicles: 96


In [37]:
model_trim_discount = (
    df
    .dropna(subset=["trim"])
    .groupby(["model", "trim"], dropna=True)
    .agg(
        vehicles=("trim", "count"),
        avg_discount=("discount_num", "mean"),
        median_discount=("discount_num", "median"),
        avg_msrp=("msrp_num", "mean"),
        avg_sale_price=("sale_price_num", "mean"),
    )
    .sort_values(["model", "avg_discount"], ascending=[True, False])
)

model_trim_discount

vehicles  avg_discount  median_discount  \
model             trim                                                    
Blazer            RS                   1           NaN              NaN   
Blazer EV         RS                   1       4500.00           4500.0   
                  SS                   1       1000.00           1000.0   
Colorado          Trail Boss           1       4500.00           4500.0   
                  WT                   1       4000.00           4000.0   
Corvette Stingray 2LT                  1           NaN              NaN   
Corvette Z06      3LZ                  1       4500.00           4500.0   
Equinox           LT                   4       3000.00           3000.0   
                  RS                   1       3000.00           3000.0   
                  ACTIV                1           NaN              NaN   
Equinox EV        RS                   3       8500.00           8500.0   
                  LT                   3       7500.00           7500.0   
Silverado 1500    ZR2                  1      11750.00          11750.0   
                  RST                  4      11562.50          11250.0   
                  LT                   4      11250.00          11250.0   
                  Custom               4      10000.00          10000.0   
                  WT                   4       7937.50           9000.0   
                  LTZ                  1       4250.00           4250.0   
Silverado 2500 HD High Country         1       7500.00           7500.0   
                  LTZ                  2       7000.00           7000.0   
                  WT                   3       6500.00           6000.0   
Silverado 3500 HD LT                   1       1000.00           1000.0   
Suburban          Premier              1       6052.00           6052.0   
                  LT                   1       3788.00           3788.0   
Trailblazer       ACTIV                2       2201.50           2201.5   
                  RS                   2       1243.00           1243.0   
                  LT                   4       1105.25           1102.5   
                  LS                   2       1052.00           1052.0   
Traverse          High Country         1       4500.00           4500.0   
                  RS                   2       4500.00           4500.0   
                  Z71                  3       4500.00           4500.0   
                  LT                   4       3937.50           3750.0   
Trax              2RS                  7       1027.00           1027.0   
                  LS                   2        959.00            959.0   
                  1RS                  1           NaN              NaN   
                  LT                   7           NaN              NaN   

                                     avg_msrp  avg_sale_price  
model             trim                                         
Blazer            RS             52515.000000             NaN  
Blazer EV         RS             55180.000000    50680.000000  
                  SS             66780.000000    65780.000000  
Colorado          Trail Boss     48935.000000    44435.000000  
                  WT             37730.000000    33730.000000  
Corvette Stingray 2LT            91210.000000             NaN  
Corvette Z06      3LZ           147020.000000   142520.000000  
Equinox           LT             34201.250000    31201.250000  
                  RS             35720.000000    32720.000000  
                  ACTIV          38610.000000             NaN  
Equinox EV        RS             50693.333333    42193.333333  
                  LT             39108.333333    31608.333333  
Silverado 1500    ZR2            83790.000000    72040.000000  
                  RST            65120.000000    53557.500000  
                  LT             61772.500000    50522.500000  
                  Custom         50361.250000    40361.250000  
                  WT         

In [38]:
df["discount_pct"] = df["discount_num"] / df["msrp_num"]

model_discount_pct = (
    df
    .groupby("model", dropna=True)
    .agg(
        vehicles=("model", "count"),
        avg_discount_pct=("discount_pct", "mean"),
        median_discount_pct=("discount_pct", "median"),
    )
    .sort_values("avg_discount_pct", ascending=False)
)

model_discount_pct


,vehicles,avg_discount_pct,median_discount_pct
model,,,
Equinox EV,7,0.179216,0.176020
Silverado 1500,20,0.179110,0.183449
Silverado 2500 HD,6,0.100177,0.089968
Colorado,2,0.098988,0.098988
Equinox,10,0.093797,0.088274
Traverse,11,0.081548,0.078215
Express Cargo 2500,1,0.061343,0.061343
Suburban,2,0.059975,0.059975
Blazer EV,2,0.048263,0.048263


In [39]:
df[
    (df["discount_num"] < 0) |
    (df["discount_num"] > df["msrp_num"] * 0.5)
][["model", "trim", "msrp", "sale_price", "discount"]]


,model,trim,msrp,sale_price,discount


In [40]:
count_by_model = (
    df
    .groupby("model", dropna=True)
    .size()
    .rename("count")
    .sort_values(ascending=False)
)

count_by_model


model
Silverado 1500        20
Trax                  18
Trailblazer           13
Traverse              11
Equinox               10
Equinox EV             7
Silverado 2500 HD      6
Blazer EV              2
Colorado               2
Suburban               2
Blazer                 1
Corvette Stingray      1
Corvette Z06           1
Express Cargo 2500     1
Silverado 3500 HD      1
Name: count, dtype: int64